In [ ]:
def calcRomanAngles(target, ts, r_obs_G, r_sun_G=None):
    from astropy.coordinates import SkyCoord, BarycentricMeanEcliptic
    import numpy as np
    import astropy.units as u

    if r_sun_G is None:
        r_sun_G = getSunPositions(ts)

    r_sun_obs = r_sun_G - r_obs_G
    rhat_sun_obs = (r_sun_obs / np.linalg.norm(r_sun_obs, axis=0)).value

    # Attempt to propagate motion
    try:
        target_updated = target.apply_space_motion(new_obstime=ts)
    except ValueError:
        target_updated = target

    # Use distance if available, otherwise set default
    if hasattr(target_updated, 'distance') and target_updated.distance is not None:
        distance = target_updated.distance
    else:
        distance = 1 * u.pc

    # Transform target to BarycentricMeanEcliptic with valid distance
    r_target_G = SkyCoord(
        ra=target_updated.icrs.ra,
        dec=target_updated.icrs.dec,
        distance=distance,
        frame="icrs",
        obstime=ts
    ).transform_to(BarycentricMeanEcliptic()).cartesian.xyz

    # Fallback: ensure result has length units
    if not hasattr(r_target_G, 'unit') or r_target_G.unit == u.dimensionless_unscaled:
        r_target_G = r_target_G * distance.to(u.AU) / distance

    r_target_obs = r_target_G - r_obs_G
    rhat_target_obs = (r_target_obs / np.linalg.norm(r_target_obs, axis=0)).value

    sun_ang = (
        np.arccos([np.dot(x, y) for x, y in zip(rhat_sun_obs.T, rhat_target_obs.T)])
        * u.rad
    )

    e2 = np.array([0, 1, 0])
    e3 = np.array([0, 0, 1])

    r_sun_obs_proj1 = projplane(r_sun_obs, e2)
    rhat_sun_obs_proj1 = (
        r_sun_obs_proj1 / np.linalg.norm(r_sun_obs_proj1, axis=0)
    ).value
    ang1 = np.array([calcang(x, e3, e2) for x in rhat_sun_obs_proj1.T])
    B_C_I = np.dstack([rotMat(2, -a) for a in ang1])

    b_3 = B_C_I[2, :, :].T
    b_1 = B_C_I[0, :, :].T
    ang2 = np.array([calcang(x, b3, b1) for x, b3, b1 in zip(rhat_sun_obs.T, b_3, b_1)])

    B_C_I = np.dstack(
        [np.matmul(rotMat(1, -a), B_C_I[:, :, j]) for j, a in enumerate(ang2)]
    )

    r_target_obs_proj1 = np.hstack(
        [
            projplane(np.array(r_target_obs[:, j], ndmin=2).T, B_C_I[2, :, j].T)
            for j in range(len(ts))
        ]
    )
    rhat_target_obs_proj1 = r_target_obs_proj1 / np.linalg.norm(
        r_target_obs_proj1, axis=0
    )

    b_1 = B_C_I[0, :, :].T
    b_3 = B_C_I[2, :, :].T
    yaw = -np.array(
        [calcang(x, b1, b3) for x, b1, b3 in zip(rhat_target_obs_proj1.T, b_1, b_3)]
    )

    B_C_I = np.dstack(
        [np.matmul(rotMat(3, a), B_C_I[:, :, j]) for j, a in enumerate(yaw)]
    )

    b_1 = B_C_I[0, :, :].T
    b_2 = B_C_I[1, :, :].T
    pitch = -np.array(
        [calcang(x, b1, b2) for x, b1, b2 in zip(rhat_target_obs.T, b_1, b_2)]
    )

    B_C_I = np.dstack(
        [np.matmul(rotMat(2, a), B_C_I[:, :, j]) for j, a in enumerate(pitch)]
    )

    return sun_ang, yaw * u.rad, pitch * u.rad, B_C_I
